# Week 2 Notebook: Inspect, Validate and Profile the Dataset

**Team:** Kayla Amani, Elisha Juma, Chemi Azikiwe, Mike Gichimu, Ruvarashe Mandima
**Dataset:** `agri_merged_raw.csv` = FAOSTAT crop data (base) merged with the Kaggle `yield_df.csv` climate data (see README for the merge method)

**Rule for this week:** we only *look*. Nothing in the source data is changed or saved over. Cleaning decisions belong to Week 3.

Sections
1. Load the data (relative paths)
2. First look: head, tail, shape, info, dtypes, describe
3. Data dictionary
4. Reusable functions
5. Quality audit: nulls, duplicates, ranges, categories, dates, cross-checks
6. Audit table
7. Five observations and three data-quality issues

## 1. Load the data

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# works from notebooks/ or from the project root
for p in ["../data/raw/agri_merged_raw.csv", "data/raw/agri_merged_raw.csv"]:
    if Path(p).exists():
        DATA_PATH = p
        break
else:
    raise FileNotFoundError("agri_merged_raw.csv not found in data/raw/")

df = pd.read_csv(DATA_PATH, parse_dates=["observation_date"])
print("Loaded:", DATA_PATH)

Loaded: ../data/raw/agri_merged_raw.csv


## 2. First look

In [2]:
print("Shape (rows, columns):", df.shape)
print("Data rows (header excluded):", len(df), "| >= 15,000:", len(df) >= 15000)
print("Columns:", df.shape[1], "| >= 10:", df.shape[1] >= 10)

Shape (rows, columns): (25432, 14)
Data rows (header excluded): 25432 | >= 15,000: True
Columns: 14 | >= 10: True


In [3]:
df.head(10)

,record_id,observation_date,year,country,iso3,region,crop,plot_area_ha,production_tonnes,yield_kg_ha,yield_hg_ha,rainfall_mm,pesticides_tonnes,temperature_c
0,1,1990-01-01,1990,Afghanistan,AFG,Southern Asia,Maize (corn),273000.0,480000.0,1758.2,NaN,NaN,NaN,NaN
1,2,1991-01-01,1991,Afghanistan,AFG,Southern Asia,Maize (corn),250000.0,420000.0,1680.0,NaN,NaN,NaN,NaN
2,3,1992-01-01,1992,Afghanistan,AFG,Southern Asia,Maize (corn),200000.0,300000.0,1500.0,NaN,NaN,NaN,NaN
3,4,1993-01-01,1993,Afghanistan,AFG,Southern Asia,Maize (corn),280000.0,470000.0,1678.6,NaN,NaN,NaN,NaN
4,5,1994-01-01,1994,Afghanistan,AFG,Southern Asia,Maize (corn),300000.0,500000.0,1666.7,NaN,NaN,NaN,NaN
5,6,1995-01-01,1995,Afghanistan,AFG,Southern Asia,Maize (corn),320000.0,530000.0,1656.3,NaN,NaN,NaN,NaN
6,7,1996-01-01,1996,Afghanistan,AFG,Southern Asia,Maize (corn),220000.0,360000.0,1636.4,NaN,NaN,NaN,NaN
7,8,1997-01-01,1997,Afghanistan,AFG,Southern Asia,Maize (corn),200000.0,300000.0,1500.0,NaN,NaN,NaN,NaN
8,9,1998-01-01,1998,Afghanistan,AFG,Southern Asia,Maize (corn),200000.0,330000.0,1650.0,NaN,NaN,NaN,NaN
9,10,1999-01-01,1999,Afghanistan,AFG,Southern Asia,Maize (corn),160000.0,240000.0,1500.0,NaN,NaN,NaN,NaN


In [4]:
df.tail(5)

,record_id,observation_date,year,country,iso3,region,crop,plot_area_ha,production_tonnes,yield_kg_ha,yield_hg_ha,rainfall_mm,pesticides_tonnes,temperature_c
25427,25428,2009-01-01,2009,Zimbabwe,ZWE,Eastern Africa,Wheat,4000.0,12000.00,3000.0,30000.0,657.0,3269.99,20.52
25428,25429,2010-01-01,2010,Zimbabwe,ZWE,Eastern Africa,Wheat,6478.0,18000.00,2778.8,27681.0,657.0,3305.17,21.17
25429,25430,2011-01-01,2011,Zimbabwe,ZWE,Eastern Africa,Wheat,8645.0,23000.00,2660.4,26274.0,657.0,3340.35,20.78
25430,25431,2012-01-01,2012,Zimbabwe,ZWE,Eastern Africa,Wheat,21377.0,53231.78,2490.2,24420.0,657.0,3375.53,20.52
25431,25432,2013-01-01,2013,Zimbabwe,ZWE,Eastern Africa,Wheat,21753.0,50966.05,2343.0,22888.0,657.0,2550.07,19.76


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25432 entries, 0 to 25431
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   record_id          25432 non-null  int64         
 1   observation_date   25432 non-null  datetime64[us]
 2   year               25432 non-null  int64         
 3   country            25432 non-null  str           
 4   iso3               25432 non-null  str           
 5   region             25432 non-null  str           
 6   crop               25432 non-null  str           
 7   plot_area_ha       25430 non-null  float64       
 8   production_tonnes  25432 non-null  float64       
 9   yield_kg_ha        25219 non-null  float64       
 10  yield_hg_ha        13104 non-null  float64       
 11  rainfall_mm        13104 non-null  float64       
 12  pesticides_tonnes  13104 non-null  float64       
 13  temperature_c      13104 non-null  float64       
dtypes: datetime64[us]

In [6]:
df.dtypes

record_id                     int64
observation_date     datetime64[us]
year                          int64
country                         str
iso3                            str
region                          str
crop                            str
plot_area_ha                float64
production_tonnes           float64
yield_kg_ha                 float64
yield_hg_ha                 float64
rainfall_mm                 float64
pesticides_tonnes           float64
temperature_c               float64
dtype: object

In [7]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
record_id,25432.0,NaN,NaN,NaN,12716.5,1.0,6358.75,12716.5,19074.25,25432.0,7341.730359
observation_date,25432,NaN,NaN,NaN,2001-08-04 11:15:19.534444,1990-01-01 00:00:00,1996-01-01 00:00:00,2002-01-01 00:00:00,2008-01-01 00:00:00,2013-01-01 00:00:00,NaN
year,25432.0,NaN,NaN,NaN,2001.590948,1990.0,1996.0,2002.0,2008.0,2013.0,6.882002
country,25432,204,Cameroon,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iso3,25432,201,CMR,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,25432,26,Eastern Africa,2839,NaN,NaN,NaN,NaN,NaN,NaN,NaN
crop,25432,10,Maize (corn),3838,NaN,NaN,NaN,NaN,NaN,NaN,NaN
plot_area_ha,25430.0,NaN,NaN,NaN,657928.232639,0.0,1548.25,23194.5,164662.75,48178000.0,3047134.104182
production_tonnes,25432.0,NaN,NaN,NaN,2668640.585816,0.0,6126.875,93000.0,715407.0,351271870.0,13929154.236324
yield_kg_ha,25219.0,NaN,NaN,NaN,6873.871533,0.0,1758.5,3955.4,9500.0,55485.5,7524.272739


## 3. Data dictionary
The full version is saved in `docs/data_dictionary.md`. This table is the same information in the notebook.

In [8]:
data_dictionary = pd.DataFrame([
    ("record_id", "int", "Unique row number (1 to 25,432)", "none", "none"),
    ("observation_date", "datetime", "1 January of the observation year, built from `year`", "date", "none"),
    ("year", "int", "Calendar year of the observation", "year (1990-2013)", "none"),
    ("country", "string", "Country name as written in FAOSTAT", "204 names", "none"),
    ("iso3", "string", "ISO 3-letter country code, added with country_converter", "3-letter code", "none"),
    ("region", "string", "UN sub-region of the country, added with country_converter", "26 labels", "none"),
    ("crop", "string", "Crop name as written in FAOSTAT", "10 crops", "none"),
    ("plot_area_ha", "float", "Area harvested (FAOSTAT)", "hectares", "2 rows (plus 242 zeros)"),
    ("production_tonnes", "float", "Crop production (FAOSTAT)", "tonnes", "none (213 zeros)"),
    ("yield_kg_ha", "float", "Yield per hectare reported by FAOSTAT", "kg per hectare", "213 rows (211 have zero area)"),
    ("yield_hg_ha", "float", "Yield from the Kaggle file (original unit)", "hectograms per hectare (divide by 10 for kg/ha)", "12,328 rows (no Kaggle match)"),
    ("rainfall_mm", "float", "Average rainfall for the country, one fixed value per country (same in every year) (Kaggle file)", "mm per year", "12,328 rows (no Kaggle match)"),
    ("pesticides_tonnes", "float", "Pesticide use for the country-year, same for all crops (Kaggle file)", "tonnes", "12,328 rows (no Kaggle match)"),
    ("temperature_c", "float", "Average annual temperature for the country (Kaggle file)", "degrees Celsius", "12,328 rows (no Kaggle match)"),
], columns=["column", "type", "meaning", "units / categories", "expected missing"])
data_dictionary

,column,type,meaning,units / categories,expected missing
0,record_id,int,"Unique row number (1 to 25,432)",none,none
1,observation_date,datetime,"1 January of the observation year, built from ...",date,none
2,year,int,Calendar year of the observation,year (1990-2013),none
3,country,string,Country name as written in FAOSTAT,204 names,none
4,iso3,string,"ISO 3-letter country code, added with country_...",3-letter code,none
5,region,string,"UN sub-region of the country, added with count...",26 labels,none
6,crop,string,Crop name as written in FAOSTAT,10 crops,none
7,plot_area_ha,float,Area harvested (FAOSTAT),hectares,2 rows (plus 242 zeros)
8,production_tonnes,float,Crop production (FAOSTAT),tonnes,none (213 zeros)
9,yield_kg_ha,float,Yield per hectare reported by FAOSTAT,kg per hectare,213 rows (211 have zero area)


## 4. Reusable functions
- `validate_columns()` checks that the expected columns exist with the right type.
- `summarize_missingness()` returns null counts and percentages, worst first.
- `check_ranges()` counts rows that break a simple rule (used in the range audit).

In [9]:
def validate_columns(data, expected):
    """Compare a DataFrame with {column: dtype kind} (kind: i, f, O/str, M).
    Returns a table with one row per expected column and prints any extra columns."""
    kinds = {"int": "i", "float": "f", "string": "O", "datetime": "M"}
    rows = []
    for col, want in expected.items():
        present = col in data.columns
        got = data[col].dtype.kind if present else None
        if got == "O" or str(data[col].dtype if present else "") in ("str", "string"):
            got = "O"
        rows.append({"column": col, "present": present,
                     "expected_type": want,
                     "actual_dtype": str(data[col].dtype) if present else "missing",
                     "ok": present and got == kinds[want]})
    extra = [c for c in data.columns if c not in expected]
    if extra:
        print("Extra columns not in the expected list:", extra)
    return pd.DataFrame(rows)


def summarize_missingness(data):
    """Null count and percentage per column, worst first."""
    out = pd.DataFrame({"missing": data.isna().sum(),
                        "missing_pct": (data.isna().mean() * 100).round(2)})
    return out.sort_values("missing", ascending=False)


def check_ranges(data, rules):
    """rules = {description: boolean Series that is True for a BAD row}. Returns counts."""
    return pd.DataFrame({"rule": list(rules), "bad_rows": [int(v.sum()) for v in rules.values()]})

In [10]:
expected = {"record_id": "int", "observation_date": "datetime", "year": "int",
            "country": "string", "iso3": "string", "region": "string", "crop": "string",
            "plot_area_ha": "float", "production_tonnes": "float", "yield_kg_ha": "float",
            "yield_hg_ha": "float", "rainfall_mm": "float", "pesticides_tonnes": "float",
            "temperature_c": "float"}
validate_columns(df, expected)

,column,present,expected_type,actual_dtype,ok
0,record_id,True,int,int64,True
1,observation_date,True,datetime,datetime64[us],True
2,year,True,int,int64,True
3,country,True,string,str,True
4,iso3,True,string,str,True
5,region,True,string,str,True
6,crop,True,string,str,True
7,plot_area_ha,True,float,float64,True
8,production_tonnes,True,float,float64,True
9,yield_kg_ha,True,float,float64,True


In [11]:
summarize_missingness(df)

,missing,missing_pct
pesticides_tonnes,12328,48.47
temperature_c,12328,48.47
yield_hg_ha,12328,48.47
rainfall_mm,12328,48.47
yield_kg_ha,213,0.84
plot_area_ha,2,0.01
region,0,0.00
iso3,0,0.00
country,0,0.00
year,0,0.00


## 5. Quality audit

### 5.1 Missing values

In [12]:
miss = summarize_missingness(df)
miss[miss["missing"] > 0]

,missing,missing_pct
pesticides_tonnes,12328,48.47
temperature_c,12328,48.47
yield_hg_ha,12328,48.47
rainfall_mm,12328,48.47
yield_kg_ha,213,0.84
plot_area_ha,2,0.01


The four climate columns all have the same number of nulls. Check whether they are missing in the *same rows* (which would mean one cause, not four).

In [13]:
climate = ["yield_hg_ha", "rainfall_mm", "pesticides_tonnes", "temperature_c"]
same_rows = df[climate].isna().all(axis=1).sum()
any_rows = df[climate].isna().any(axis=1).sum()
print("Rows where ALL four are null:", same_rows)
print("Rows where ANY of the four is null:", any_rows)
print("Rows complete in every column:", len(df.dropna()))

Rows where ALL four are null: 12328
Rows where ANY of the four is null: 12328
Rows complete in every column: 13101


In [14]:
no_climate = df["rainfall_mm"].isna()
print("Countries with climate data:   ", df.loc[~no_climate, "country"].nunique())
print("Countries with NO climate data:", df[df.groupby("country")["rainfall_mm"].transform(lambda s: s.isna().all())]["country"].nunique())
print()
print("Null % of rainfall by year (first and last years, and 2003):")
by_year = (df.groupby("year")["rainfall_mm"].apply(lambda s: s.isna().mean() * 100)).round(1)
print(by_year.loc[[1990, 2002, 2003, 2004, 2013]].to_string())

Countries with climate data:    101
Countries with NO climate data: 103

Null % of rainfall by year (first and last years, and 2003):
year
1990     47.4
2002     46.5
2003    100.0
2004     46.4
2013     45.0


In [15]:
print("Null % of rainfall by crop:")
print((df.groupby("crop")["rainfall_mm"].apply(lambda s: s.isna().mean() * 100)).round(1).sort_values().to_string())
print()
print("Null % of rainfall by region (worst 8):")
print((df.groupby("region")["rainfall_mm"].apply(lambda s: s.isna().mean() * 100)).round(1).sort_values(ascending=False).head(8).to_string())

Null % of rainfall by crop:
crop
Wheat                            38.1
Potatoes                         42.6
Soya beans                       43.4
Sorghum                          43.4
Rice                             46.6
Maize (corn)                     47.7
Sweet potatoes                   56.3
Plantains and cooking bananas    59.9
Cassava, fresh                   60.3
Yams                             65.2

Null % of rainfall by region (worst 8):
region
Belgium-Luxembourg       100.0
Micronesia               100.0
USSR                     100.0
Yugoslav SFR             100.0
Polynesia                100.0
Serbia and Montenegro    100.0
Eastern Asia              83.4
Melanesia                 75.2


Now the FAOSTAT-side nulls (`yield_kg_ha`, `plot_area_ha`).

In [16]:
y_null = df[df["yield_kg_ha"].isna()]
print("yield_kg_ha nulls:", len(y_null))
print("  of which plot_area_ha == 0:", int((y_null["plot_area_ha"] == 0).sum()))
print("  of which production_tonnes == 0:", int((y_null["production_tonnes"] == 0).sum()))
print()
print("plot_area_ha nulls:", int(df["plot_area_ha"].isna().sum()))
df[df["plot_area_ha"].isna()][["country", "crop", "year", "plot_area_ha", "production_tonnes", "yield_kg_ha"]]

yield_kg_ha nulls: 213
  of which plot_area_ha == 0: 211
  of which production_tonnes == 0: 212

plot_area_ha nulls: 2


,country,crop,year,plot_area_ha,production_tonnes,yield_kg_ha
9545,Grenada,"Cassava, fresh",2013,NaN,0.0,NaN
9641,Grenada,Yams,2013,NaN,0.0,NaN


### 5.2 Duplicates

In [17]:
print("Fully duplicated rows:", int(df.duplicated().sum()))
print("Duplicate country-crop-year keys:", int(df.duplicated(["country", "crop", "year"]).sum()))
print("Duplicate record_id:", int(df["record_id"].duplicated().sum()))

Fully duplicated rows: 0
Duplicate country-crop-year keys: 0
Duplicate record_id: 0


### 5.3 Invalid or suspicious numeric ranges

In [18]:
rules = {
    "rainfall_mm < 0": df["rainfall_mm"] < 0,
    "temperature_c outside -60..60": ~df["temperature_c"].dropna().between(-60, 60).reindex(df.index, fill_value=True),
    "plot_area_ha < 0": df["plot_area_ha"] < 0,
    "plot_area_ha == 0": df["plot_area_ha"] == 0,
    "production_tonnes < 0": df["production_tonnes"] < 0,
    "production_tonnes == 0": df["production_tonnes"] == 0,
    "yield_kg_ha == 0 (reported, not null)": df["yield_kg_ha"] == 0,
    "pesticides_tonnes < 0": df["pesticides_tonnes"] < 0,
}
range_audit = check_ranges(df, rules)
range_audit

,rule,bad_rows
0,rainfall_mm < 0,0
1,temperature_c outside -60..60,0
2,plot_area_ha < 0,0
3,plot_area_ha == 0,242
4,production_tonnes < 0,0
5,production_tonnes == 0,213
6,"yield_kg_ha == 0 (reported, not null)",1
7,pesticides_tonnes < 0,0


In [19]:
num = ["plot_area_ha", "production_tonnes", "yield_kg_ha", "rainfall_mm", "pesticides_tonnes", "temperature_c"]
df[num].describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
plot_area_ha,25430.0,657928.2,3047134.1,0.0,1548.2,23194.5,164662.8,48178000.0
production_tonnes,25432.0,2668640.6,13929154.2,0.0,6126.9,93000.0,715407.0,351271870.0
yield_kg_ha,25219.0,6873.9,7524.3,0.0,1758.5,3955.4,9500.0,55485.5
rainfall_mm,13104.0,1154.2,740.9,51.0,608.0,1083.0,1622.0,3240.0
pesticides_tonnes,13104.0,14789.8,33663.9,0.0,264.5,2152.0,13278.5,367778.0
temperature_c,13104.0,19.8,6.7,1.3,15.7,20.6,25.8,30.4


In [20]:
print("Rows with plot_area_ha == 0 but a yield still reported:", int(((df["plot_area_ha"] == 0) & df["yield_kg_ha"].notna()).sum()))
print()
print("Yield (kg/ha) by crop, to look for silly values:")
print(df.groupby("crop")["yield_kg_ha"].agg(["count", "min", "median", "max"]).round(1).to_string())
print()
df.nlargest(5, "yield_kg_ha")[["country", "crop", "year", "yield_kg_ha"]]

Rows with plot_area_ha == 0 but a yield still reported: 31

Yield (kg/ha) by crop, to look for silly values:
                               count     min   median      max
crop                                                          
Cassava, fresh                  2361  1091.4   9599.8  43667.9
Maize (corn)                    3804    84.9   2273.4  36761.9
Plantains and cooking bananas   1147    28.6   8156.0  55485.5
Potatoes                        3635   840.6  15228.4  52229.1
Rice                            2753   285.7   3064.6  10389.5
Sorghum                         2502     0.0   1249.1  35720.2
Soya beans                      2154     5.0   1466.7   4160.9
Sweet potatoes                  2650   221.1   7497.0  40000.0
Wheat                           2909   131.4   2313.1   9938.7
Yams                            1304   136.7   7659.3  33333.3



,country,crop,year,yield_kg_ha
1819,Belize,Plantains and cooking bananas,2009,55485.5
1729,Belgium-Luxembourg,Potatoes,1997,52229.1
1810,Belize,Plantains and cooking bananas,2000,50707.4
1695,Belgium,Potatoes,2011,50141.2
1725,Belgium-Luxembourg,Potatoes,1993,49937.6


### 5.4 Inconsistent categories

In [21]:
print("Crop values:")
print(df["crop"].value_counts().to_string())
print()
print("Leading/trailing spaces -> crop:", int((df["crop"] != df["crop"].str.strip()).sum()),
      "| country:", int((df["country"] != df["country"].str.strip()).sum()),
      "| region:", int((df["region"] != df["region"].str.strip()).sum()))

Crop values:
crop
Maize (corn)                     3838
Potatoes                         3646
Wheat                            2923
Rice                             2811
Sweet potatoes                   2664
Sorghum                          2537
Cassava, fresh                   2368
Soya beans                       2193
Yams                             1305
Plantains and cooking bananas    1147

Leading/trailing spaces -> crop: 0 | country: 0 | region: 0


In [22]:
print("Number of region labels:", df["region"].nunique())
region_counts = df["region"].value_counts()
region_counts.tail(8)

Number of region labels: 26


region
Polynesia                    331
Australia and New Zealand    267
Northern America             264
Micronesia                   118
Serbia and Montenegro         70
Belgium-Luxembourg            40
USSR                          12
Yugoslav SFR                  12
Name: count, dtype: int64

In [23]:
# region labels that are really former / combined countries (not UN regions)
former = ["USSR", "Yugoslav SFR", "Serbia and Montenegro", "Belgium-Luxembourg"]
print("Rows whose region is a former state name:", int(df["region"].isin(former).sum()))
print()
# different country names that share one ISO3 code
shared = df.groupby("iso3")["country"].unique()
shared[shared.apply(len) > 1]

Rows whose region is a former state name: 134



iso3
CZE    [Czechia, Czechoslovakia]
ETH     [Ethiopia, Ethiopia PDR]
SDN      [Sudan, Sudan (former)]
Name: country, dtype: object

In [24]:
print("Country names with brackets or commas:")
print([c for c in sorted(df["country"].unique()) if "(" in c or "," in c])

Country names with brackets or commas:
['Bolivia (Plurinational State of)', 'China, Hong Kong SAR', 'China, Taiwan Province of', 'China, mainland', 'Iran (Islamic Republic of)', 'Micronesia (Federated States of)', 'Netherlands (Kingdom of the)', 'Sudan (former)', 'Venezuela (Bolivarian Republic of)']


### 5.5 Date parsing

In [25]:
print("observation_date dtype:", df["observation_date"].dtype)
print("Missing dates:", int(df["observation_date"].isna().sum()))
print("Range:", df["observation_date"].min().date(), "to", df["observation_date"].max().date())
print("Every date is 1 January:", bool(((df["observation_date"].dt.month == 1) & (df["observation_date"].dt.day == 1)).all()))
print("observation_date year matches `year` column:", bool((df["observation_date"].dt.year == df["year"]).all()))
print()
print("Rows per year:")
df["year"].value_counts().sort_index().to_string()

observation_date dtype: datetime64[us]
Missing dates: 0
Range: 1990-01-01 to 2013-01-01
Every date is 1 January: True
observation_date year matches `year` column: True

Rows per year:


'year\n1990     980\n1991     980\n1992    1058\n1993    1056\n1994    1060\n1995    1066\n1996    1066\n1997    1065\n1998    1064\n1999    1063\n2000    1068\n2001    1069\n2002    1069\n2003    1072\n2004    1075\n2005    1076\n2006    1079\n2007    1063\n2008    1065\n2009    1064\n2010    1066\n2011    1066\n2012    1069\n2013    1073'

### 5.6 Cross-checks between columns and sources

In [26]:
# 1) does FAOSTAT yield equal production / area?
ok = df.dropna(subset=["yield_kg_ha", "plot_area_ha"])
ok = ok[ok["plot_area_ha"] > 0]
calc = ok["production_tonnes"] * 1000 / ok["plot_area_ha"]
off = (calc / ok["yield_kg_ha"] - 1).abs() > 0.02
print("yield_kg_ha vs production/area: rows more than 2% apart =", int(off.sum()), "of", len(ok))

yield_kg_ha vs production/area: rows more than 2% apart = 129 of 25188


In [27]:
# 2) do the Kaggle yield (hg/ha) and FAOSTAT yield (kg/ha) agree after the unit change?
both = df.dropna(subset=["yield_hg_ha", "yield_kg_ha"])
ratio = (both["yield_hg_ha"] / 10) / both["yield_kg_ha"]
disagree = (ratio - 1).abs() >= 0.01
print("Rows with both yields:", len(both))
print("Rows that differ by 1% or more:", int(disagree.sum()), f"({disagree.mean() * 100:.1f}%)")
both.loc[disagree, ["country", "crop", "year", "yield_kg_ha", "yield_hg_ha"]].assign(
    kaggle_kg_ha=lambda d: d["yield_hg_ha"] / 10).head(8)

Rows with both yields: 13101
Rows that differ by 1% or more: 712 (5.4%)


,country,crop,year,yield_kg_ha,yield_hg_ha,kaggle_kg_ha
452,Angola,Sweet potatoes,1997,7904.9,78098.0,7809.8
453,Angola,Sweet potatoes,1998,7745.2,76256.0,7625.6
492,Angola,Wheat,2013,999.1,9022.0,902.2
656,Argentina,Potatoes,2009,29332.6,284439.0,28443.9
657,Argentina,Potatoes,2010,29620.3,289849.0,28984.9
658,Argentina,Potatoes,2011,29864.9,286657.0,28665.7
659,Argentina,Potatoes,2012,30396.7,287821.0,28782.1
660,Argentina,Potatoes,2013,30735.7,288984.0,28898.4


In [28]:
# 3) climate values should be the same for every crop within one country-year
m = df.dropna(subset=["rainfall_mm"])
for c in ["rainfall_mm", "pesticides_tonnes", "temperature_c"]:
    n = int((m.groupby(["country", "year"])[c].nunique() > 1).sum())
    print(f"{c}: country-years where it varies across crops = {n}")

rainfall_mm: country-years where it varies across crops = 0
pesticides_tonnes: country-years where it varies across crops = 0
temperature_c: country-years where it varies across crops = 0


In [29]:
# 4) is rainfall really a yearly measurement? Count countries whose rainfall changes between years
r = df.dropna(subset=["rainfall_mm"])
varies = (r.groupby("country")["rainfall_mm"].nunique() > 1).sum()
print("Countries with rainfall data:", r["country"].nunique())
print("Countries where rainfall_mm changes between years:", int(varies))
print("Countries where temperature_c changes between years:", int((r.groupby("country")["temperature_c"].nunique() > 1).sum()))
print("Countries where pesticides_tonnes changes between years:", int((r.groupby("country")["pesticides_tonnes"].nunique() > 1).sum()))

Countries with rainfall data: 101
Countries where rainfall_mm changes between years: 0
Countries where temperature_c changes between years: 101
Countries where pesticides_tonnes changes between years: 100


## 6. Quality audit table

In [30]:
audit = pd.DataFrame([
    ("Shape", "25,432 rows x 14 columns", "Pass", "Meets 15,000 x 10 minimum"),
    ("Duplicate rows", int(df.duplicated().sum()), "Pass", "No exact duplicates"),
    ("Duplicate country-crop-year keys", int(df.duplicated(["country", "crop", "year"]).sum()), "Pass", "One row per key"),
    ("Nulls: yield_hg_ha, rainfall_mm, pesticides_tonnes, temperature_c", f"{int(df['rainfall_mm'].isna().sum()):,} each ({df['rainfall_mm'].isna().mean() * 100:.1f}%)", "Issue", "Same rows; 103 of 204 countries have no climate data; none for 2003"),
    ("Nulls: yield_kg_ha", int(df["yield_kg_ha"].isna().sum()), "Issue", "211 have zero area harvested; 2 have missing area (Grenada)"),
    ("Nulls: plot_area_ha", int(df["plot_area_ha"].isna().sum()), "Minor", "Grenada, 2013, zero production"),
    ("Zero area (plot_area_ha == 0)", int((df["plot_area_ha"] == 0).sum()), "Issue", "31 of them still report a yield"),
    ("Zero production", int((df["production_tonnes"] == 0).sum()), "Issue", "Crop not harvested, or not recorded"),
    ("Negative rainfall / area / production / pesticides", 0, "Pass", "No negative values"),
    ("Temperature outside -60..60 C", 0, "Pass", "Range 1.3 to 30.4"),
    ("Yield vs production/area (2% tolerance)", int(off.sum()), "Minor", "FAOSTAT yield mostly equals production / area"),
    ("Kaggle vs FAOSTAT yield (1% tolerance)", f"{int(disagree.sum()):,} of {len(both):,} ({disagree.mean() * 100:.1f}%)", "Issue", "Two sources disagree after hg to kg conversion"),
    ("Inconsistent categories: region", f"{int(df['region'].isin(former).sum()):,} rows", "Issue", "4 labels are former states, not UN regions"),
    ("Inconsistent categories: country / ISO3", "3 shared codes", "Minor", "Czechoslovakia/Czechia, Ethiopia PDR/Ethiopia, Sudan (former)/Sudan"),
    ("Inconsistent categories: crop names", "10 values", "Minor", "Style differs (Maize (corn), Cassava, fresh); no spelling variants"),
    ("Rainfall varies by year?", "0 of 101 countries", "Issue", "rainfall_mm is one fixed value per country, not a yearly measurement"),
    ("Date parsing", "0 failed", "Pass", "All dates are 1 January; matches `year`"),
], columns=["Check", "Result", "Status", "Note"])
audit

,Check,Result,Status,Note
0,Shape,"25,432 rows x 14 columns",Pass,"Meets 15,000 x 10 minimum"
1,Duplicate rows,0,Pass,No exact duplicates
2,Duplicate country-crop-year keys,0,Pass,One row per key
3,"Nulls: yield_hg_ha, rainfall_mm, pesticides_to...","12,328 each (48.5%)",Issue,Same rows; 103 of 204 countries have no climat...
4,Nulls: yield_kg_ha,213,Issue,211 have zero area harvested; 2 have missing a...
5,Nulls: plot_area_ha,2,Minor,"Grenada, 2013, zero production"
6,Zero area (plot_area_ha == 0),242,Issue,31 of them still report a yield
7,Zero production,213,Issue,"Crop not harvested, or not recorded"
8,Negative rainfall / area / production / pestic...,0,Pass,No negative values
9,Temperature outside -60..60 C,0,Pass,Range 1.3 to 30.4


## 7. Observations and data-quality issues

### Five initial observations

1. **The dataset meets the size rules and has no repeated rows.** It has 25,432 rows and 14 columns (needs 15,000 and 10), 0 duplicate rows and 0 duplicate country-crop-year keys, covering 204 countries, 10 crops and 1990-2013 (see section 2 and 5.2).
2. **About half the rows have no climate data.** Rainfall, temperature, pesticides and the Kaggle yield are each missing in the same 12,328 rows (48.5%). Only 101 of 204 countries have climate values, so 13,101 rows are complete (section 5.1).
3. **Crop yields differ by a factor of about 12.** Median yield is highest for potatoes (about 15,228 kg/ha) and lowest for sorghum (about 1,249 kg/ha). Any comparison of regions will therefore be partly a comparison of which crops each region grows (section 5.3).
4. **Rainfall and temperature show almost no overall link to yield.** The rows with climate data show a correlation near 0.03 for rainfall and -0.13 for temperature (see below). Pooling all crops hides any pattern, so Week 3 and 4 should look crop by crop.
5. **Coverage is steady across years except for climate in 2003.** There are about 980 to 1,080 rows per year, but no row from 2003 has climate data because the Kaggle file has no 2003 (section 5.1 and 5.5).

In [31]:
print(df[["rainfall_mm", "temperature_c", "yield_kg_ha"]].corr().round(2))
print()
print("Median yield_kg_ha by crop:")
print(df.groupby("crop")["yield_kg_ha"].median().sort_values().round(0).to_string())

               rainfall_mm  temperature_c  yield_kg_ha
rainfall_mm           1.00           0.34         0.03
temperature_c         0.34           1.00        -0.13
yield_kg_ha           0.03          -0.13         1.00

Median yield_kg_ha by crop:


crop
Sorghum                           1249.0
Soya beans                        1467.0
Maize (corn)                      2273.0
Wheat                             2313.0
Rice                              3065.0
Sweet potatoes                    7497.0
Yams                              7659.0
Plantains and cooking bananas     8156.0
Cassava, fresh                    9600.0
Potatoes                         15228.0


### Three potential data-quality issues

1. **Large, non-random missing climate data.** 48.5% of rows lack all four climate columns because whole countries (and the year 2003) are absent from the Kaggle source. Filling them would mean inventing values, so Week 3 should *retain* these rows with a flag and run climate analysis only on the complete rows. Climate coverage is also uneven by region (for example 5% missing in Australia and New Zealand, 83% in Eastern Asia).
2. **Zero values that look like data but are not yields.** 242 rows have zero harvested area and 213 have zero production. These leave 213 null yields (211 where area is zero, 2 where area is missing), and 31 zero-area rows still report a yield. A zero here may mean "not grown that year" rather than a measurement, so these rows should not be averaged as if they were low yields.
3. **Two sources that disagree, plus inconsistent labels.** After converting hectograms to kilograms, the Kaggle yield and the FAOSTAT yield differ by 1% or more in 712 of 13,101 rows (5.4%). Separately, 4 region labels (USSR, Yugoslav SFR, Serbia and Montenegro, Belgium-Luxembourg) are former states rather than UN regions, and three former countries share an ISO3 code with their successor. These need a documented rule before any regional or country comparison.